In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/

/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed


In [ ]:
!pip install -q datasets

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
class ReviewPairDataset(Dataset):
    def __init__(self, user_texts, item_texts, user_ids, item_ids, ratings, tokenizer, max_length=512):
        self.user_texts = user_texts
        self.item_texts = item_texts
        self.user_ids = user_ids
        self.item_ids = item_ids
        self.ratings = ratings
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.user_texts)

    def __getitem__(self, idx):
        user_enc = self.tokenizer(self.user_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        item_enc = self.tokenizer(self.item_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        user_id = torch.tensor(self.user_ids[idx], dtype=torch.long)
        item_id = torch.tensor(self.item_ids[idx], dtype=torch.long)
        rating = torch.tensor(float(self.ratings[idx]), dtype=torch.float)

        return {
            "input_ids_user": user_enc["input_ids"].squeeze(0),
            "attention_mask_user": user_enc["attention_mask"].squeeze(0),
            "input_ids_item": item_enc["input_ids"].squeeze(0),
            "attention_mask_item": item_enc["attention_mask"].squeeze(0),
            "labels": rating,
            "user_id": user_id,
            "item_id": item_id
        }

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions  # 그대로 사용!

    # 지표 계산
    mae = mean_absolute_error(labels, preds)
    mse = mean_squared_error(labels, preds)
    rmse = np.sqrt(mse)

    epsilon = 1e-8
    mape = np.mean(np.abs((labels - preds) / (labels + epsilon))) * 100

    return {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "mape": mape
    }

# 데이터셋 로딩

In [ ]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v6_final.pkl')
print(df.shape)
df.head()

(66718, 12)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,LLaMA_Output,LLaVA_Output
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on user reviews, it appears that users a...",The image shows a hand holding a set of earrin...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on user reviews, it appears that users p...",The image shows a close-up of a person's ear w...
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on this review, it seems that users pref...","The image shows a fashion accessory, specifica..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on user reviews, it appears that users p...",The image shows a sleeveless tank top with a p...
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on the user review, it appears that the ...",The image shows a lime green t-shirt with a gr...


# Prepare Input

In [ ]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

# fit-transform
df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 실험 데이터의 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

Num_User = n_users + 1
Num_Item = n_items + 1

In [ ]:
# train, valid, test으로 데이터셋 분리
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, valid = train_test_split(train, test_size=0.125, random_state=42)

train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [ ]:
train.shape, valid.shape, test.shape

((46702, 12), (6672, 12), (13344, 12))

In [ ]:
n_users, n_items

(7449, 16099)

In [ ]:
# sparsity = 1 - (len(df) / (n_users * n_items))
# print(sparsity)

In [ ]:
# train, test, valid 변수 선언
train_user_id = train['user_id'].to_numpy()
train_item_id = train['asin'].to_numpy()
train_summary = train['LLaMA_Output']
train_image_description = train['LLaVA_Output']
train_rating = train['rating'].to_numpy()

valid_user_id = valid['user_id'].to_numpy()
valid_item_id = valid['asin'].to_numpy()
valid_summary = valid['LLaMA_Output']
valid_image_description = valid['LLaVA_Output']
valid_rating = valid['rating'].to_numpy()

test_user_id = test['user_id'].to_numpy()
test_item_id = test['asin'].to_numpy()
test_summary = test['LLaMA_Output']
test_image_description = test['LLaVA_Output']
test_rating = test['rating'].to_numpy()

# Load Model

In [ ]:
# BERT 모델 Loading
# 해당 모델은 Layer 6개인 모델

# 사용자 텍스트 처리 모델
BERT_review = AutoModel.from_pretrained("bert-base-uncased")

# 아이템 텍스트 처리 모델
BERT_image = AutoModel.from_pretrained("bert-base-uncased")

# tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Two Tower BERT

```python
MAS_Rec(
  user_id: Embedding(num_users, 64)
  item_id: Embedding(num_items, 64)
  user_model: BertModel
  item_model: BertModel
  cls_dense: MLP(768→512→256)
  dot_product: Dot([user_emb, item_emb])
)

```

In [ ]:
def make_mlp(sizes, activation=nn.ReLU):
    """
    중간 feature extractor용 MLP
    모든 Linear 뒤에 activation 함수가 붙음 (마지막 레이어 포함)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        layers.append(activation())
    return nn.Sequential(*layers)

In [ ]:
def make_last_mlp(sizes, activation=nn.ReLU):
    """
    최종 예측 MLP
    마지막 Linear 뒤에는 activation이 없음 (즉, 마지막은 Linear-only)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        if i < len(sizes) - 2:
            layers.append(activation())
    return nn.Sequential(*layers)

In [ ]:
class MAS_Rec(nn.Module):
    def __init__(self, n_users, n_items, user_model, item_model):
        super().__init__()

        # Interaction IDs
        self.user_id = nn.Embedding(n_users, 64)
        self.item_id = nn.Embedding(n_items, 64)

        # Two Tower Bert
        self.user_model = user_model
        self.item_model = item_model

        # [CLS] Dense
        self.cls_dense = make_mlp([768, 512, 256])

        # Last Dense
        self.fc_final = make_last_mlp([640, 256, 1])

    def forward(self, user_id, item_id, input_ids_user, attention_mask_user, input_ids_item, attention_mask_item, labels=None):

        # Interaction IDs
        user_emb = self.user_id(user_id)    # [batch, 64]
        item_emb = self.item_id(item_id)    # [batch, 64]

        # Two Tower BERT
        # 여기에서 input_ids는 token_id(token index)
        user_cls = self.user_model(input_ids=input_ids_user, attention_mask=attention_mask_user).last_hidden_state[:, 0, :]
        item_cls = self.item_model(input_ids=input_ids_item, attention_mask=attention_mask_item).last_hidden_state[:, 0, :]

        # [CLS] Dense
        user_cls = self.cls_dense(user_cls)
        item_cls = self.cls_dense(item_cls)

        # Concat User-Item Feat
        user_feat = torch.cat([user_emb, user_cls], dim=-1)
        item_feat = torch.cat([item_emb, item_cls], dim=-1)

        # Dot-product
        logits = torch.sum(user_feat * item_feat, dim=-1)

        if labels is not None:
            loss = nn.MSELoss()(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}

# Fine Tuning

In [ ]:
# 모델 생성
model = MAS_Rec(Num_User, Num_Item, user_model=BERT_review, item_model=BERT_image)

In [ ]:
for name, module in model.named_children():
    print(name, module)

user_id Embedding(7450, 64)
item_id Embedding(16100, 64)
user_model BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_

In [ ]:
# 실제 훈련용
train_dataset = ReviewPairDataset(train_summary, train_image_description, train_user_id, train_item_id, train_rating, tokenizer, max_length=512)
valid_dataset = ReviewPairDataset(valid_summary, valid_image_description, valid_user_id, valid_item_id, valid_rating, tokenizer, max_length=512)
test_dataset = ReviewPairDataset(test_summary, test_image_description, test_user_id, test_item_id, test_rating, tokenizer, max_length=512)

In [ ]:
# 하이퍼파라미터
num_training_samples = 46702 # 46702 # 16443

# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps = 4,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    metric_for_best_model="mse",  # ✅ compute_metrics에 꼭 있어야 함
    greater_is_better=False,
    fp16=torch.cuda.is_available(),  # ✅ 안전함
)

# ✅ Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,  # 실험용으로 train=eval
    compute_metrics=compute_metrics,
)

# ✅ 학습 시작
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse,Mape
1,9.292127,0.581449,0.652385,0.581449,0.762528,17.494673
2,0.336916,0.265094,0.311912,0.265094,0.514873,9.039278
3,0.247647,0.252500,0.302796,0.252500,0.502494,8.850536
4,0.231855,0.253294,0.303963,0.253294,0.503283,8.796123
5,0.224259,0.250902,0.302456,0.250902,0.500901,8.740966


TrainOutput(global_step=915, training_loss=2.066560843212357, metrics={'train_runtime': 2240.649, 'train_samples_per_second': 104.215, 'train_steps_per_second': 0.408, 'total_flos': 0.0, 'train_loss': 2.066560843212357, 'epoch': 5.0})

# Train

In [ ]:
# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps = 4,
    num_train_epochs=100,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10,
    metric_for_best_model="mse",  # ✅ compute_metrics에 꼭 있어야 함
    greater_is_better=False,
    fp16=torch.cuda.is_available(),  # ✅ 안전함
)

# ✅ Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,  # 실험용으로 train=eval
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]  # ✅ 여기에 patience 설정
)

# ✅ 학습 시작
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse,Mape
1,0.241767,0.269915,0.307958,0.269915,0.519533,9.155558


In [ ]:
# 평가
results = trainer.evaluate()

# 평가 결과 출력
print("Evaluation Results:")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# pd.set_option('display.max_colwidth', None)

# error_sample = df.loc[df['LLaMA_Output'].str.contains('Input')==True, ['title_y', 'text', 'LLaMA_Output']].tail(10)

In [ ]:
import numpy as np

# 데이터 입력
results = {
    "Proposed": {
        "MAE":  [0.3121, 0.3260, 0.3129, 0.3078, 0.3113],
        "MSE":  [0.2494, 0.2682, 0.2614, 0.2701, 0.2673],
        "RMSE": [0.4994, 0.5179, 0.5113, 0.5197, 0.5170],
        "MAPE": [0.0901, 0.0938, 0.0915, 0.0910, 0.0917]
    }
}


# 평균과 표준편차 계산
summary = {}
for model, metrics in results.items():
    summary[model] = {}
    for metric, values in metrics.items():
        values_arr = np.array(values)
        summary[model][metric] = {
            "mean": round(values_arr.mean(), 6),
            "std": round(values_arr.std(ddof=1), 6)  # 표본 표준편차
        }

# 결과 출력
for model, metrics in summary.items():
    print(f"Model: {model}")
    for metric, stats in metrics.items():
        print(f"  {metric} -> Mean: {stats['mean']}, Std: {stats['std']}")